In [11]:
import pandas as pd
df = pd.read_csv("dataset_heart.csv")
df.head()

,age,sex,chest pain type,resting blood pressure,serum cholestoral,fasting blood sugar,resting electrocardiographic results,max heart rate,exercise induced angina,oldpeak,ST segment,major vessels,thal,heart disease
0,70,1,4,130,322,0,2,109,0,2.4,2,3,3,2
1,67,0,3,115,564,0,2,160,0,1.6,2,0,7,1
2,57,1,2,124,261,0,0,141,0,0.3,1,0,7,2
3,64,1,4,128,263,0,0,105,1,0.2,2,1,7,1
4,74,0,2,120,269,0,2,121,1,0.2,1,1,3,1


In [12]:
 # Dropping features (X) and target (y)
 X = df.drop("heart disease", axis=1)
 y = df["heart disease"]

In [13]:
from sklearn.model_selection import train_test_split

# Spliting into train/test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(len(X_train))
print(len(X_test))

216
54


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train logistic regression model
model = LogisticRegression(max_iter=1000) # This ensures convergence
model.fit(X_train, y_train)

# Making our predicitons
y_pred = model.predict(X_test)

# Evaluting
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\n Classsification Report:\n", classification_report(y_test, y_pred))



Accuracy: 0.8518518518518519

 Classsification Report:
               precision    recall  f1-score   support

           1       0.92      0.80      0.86        30
           2       0.79      0.92      0.85        24

    accuracy                           0.85        54
   macro avg       0.85      0.86      0.85        54
weighted avg       0.86      0.85      0.85        54



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
# Using Grid Search - Menaing giving Hyperparameters basically settings we can tweak
from sklearn.model_selection import GridSearchCV

# Define parameter grid to search
param_grid = {
    "C": [0.01,0.1, 1, 10, 100],
    "penalty": ["l1", "12"],
    "solver": ["liblinear"]
}

# Grid Search
grid = GridSearchCV(model, param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

# Prints the best parameter and best score
print("Best Parameter:", grid.best_params_)
print("Best Parameter:", grid.best_score_)

# Evaluting the test data
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report\n", classification_report(y_test, y_pred))



Best Parameter: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Best Parameter: 0.8288583509513741

Test Accuracy: 0.8518518518518519

Confusion Matrix:
 [[24  6]
 [ 2 22]]

Classification Report
               precision    recall  f1-score   support

           1       0.92      0.80      0.86        30
           2       0.79      0.92      0.85        24

    accuracy                           0.85        54
   macro avg       0.85      0.86      0.85        54
weighted avg       0.86      0.85      0.85        54



/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
25 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_

In [16]:
from sklearn.svm import SVC

# Creating a SVM classifer without Grid search
svm_model = SVC(kernel= "linear", C=1)
svm_model.fit(X_train, y_train)

# Predictions
y_pred = svm_model.predict(X_test)

# Evalution
print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report\n", classification_report(y_test, y_pred))


Test Accuracy: 0.8518518518518519

Confusion Matrix:
 [[25  5]
 [ 3 21]]

Classification Report
               precision    recall  f1-score   support

           1       0.89      0.83      0.86        30
           2       0.81      0.88      0.84        24

    accuracy                           0.85        54
   macro avg       0.85      0.85      0.85        54
weighted avg       0.86      0.85      0.85        54



In [17]:
# Creating a SVM classifer with Grid search
param_grid = {
    "kernel": ["linear", "rbf", "poly"],
    "C": [0.01, 0.1, 1, 10, 100],
    "gamma": ["scale", "auto"]
}

grid = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ALWAYS scale features for SVM
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_s  = scaler.transform(X_test)        # transform test with the same scaler

# Base model (no tuning yet)
svm = SVC()   # default kernel='rbf'

# What we want GridSearch to try (small, sensible grid)
param_grid = {
    "C": [0.1, 1, 10],               # model flexibility: small=more regularization, large=less
    "kernel": ["linear", "rbf"],     # two common kernels
    "gamma": ["scale", "auto"]       # only matters for 'rbf'; ignore for 'linear'
}

# Grid search (5-fold CV), use all CPU cores with n_jobs=-1
grid = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train the grid on the *scaled* training data
grid.fit(X_train_s, y_train)

# Best combo found on cross-validation
print("Best params:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 3))

# Evaluate the best model on the test set
best_svm = grid.best_estimator_
y_pred = best_svm.predict(X_test_s)

print("\nTest Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Best params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Best CV accuracy: 0.838

Test Accuracy: 0.852

Confusion Matrix:
 [[25  5]
 [ 3 21]]

Classification Report
               precision    recall  f1-score   support

           1       0.89      0.83      0.86        30
           2       0.81      0.88      0.84        24

    accuracy                           0.85        54
   macro avg       0.85      0.85      0.85        54
weighted avg       0.86      0.85      0.85        54



In [38]:
import xgboost as xgb
from xgboost import XGBClassifier
import numpy as np

# XGBoost without Grid Search

# Map target variable classes to 0 and 1
y_train_bin = (y_train - 1).astype(int)
y_test_bin = (y_test - 1).astype(int)

# y_train_bin = (y_train == 2).astype(int)
# y_test_bin = (y_test == 2 ).astype(int)

# Creating XGBoost model with manual hyperparameters
xgb_model = XGBClassifier(
    n_estimators=100,          # number of boosting rounds (i.e., trees)
    max_depth=3,               # max depth of each tree (model complexity)
    learning_rate=0.1,         # step size shrinkage per tree (smaller = safer/slower)
    subsample=0.8,             # fraction of rows sampled per tree (regularization)
    colsample_bytree=0.8,      # fraction of features sampled per tree (regularization)
    random_state=42,           # reproducibility
    use_label_encoder=False,   # disable legacy label encoder
    eval_metric="logloss"      # metric for binary classification (lower is better)
)


# Fit model
xgb_model.fit(X_train, y_train_bin)

# Predict
y_pred = xgb_model.predict(X_test)

# Evalute
print("\nAcccuracy Score:\n", accuracy_score(y_test_bin, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_bin, y_pred))
print("\nClassification Report:\n", classification_report(y_test_bin, y_pred))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [12:14:19] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Acccuracy Score:
 0.7962962962962963

Confusion Matrix:
 [[23  7]
 [ 4 20]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.77      0.81        30
           1       0.74      0.83      0.78        24

    accuracy                           0.80        54
   macro avg       0.80      0.80      0.80        54
weighted avg       0.80      0.80      0.80        54



In [39]:
# XGBoost with Grid Search

# Base model (turn off built-in label encoding; set a metric)
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=42,
    nthread=-1
)

param_grid = {
    "learning_rate": [0.01, 0.1, 0.2],   # step size per tree
    "n_estimators":  [100, 300, 600],    # number of trees (more = slower, often better)
    "max_depth":     [3, 5, 7],          # tree depth (higher = more complex)
    "subsample":     [0.7, 1.0],         # row sampling per tree
    "colsample_bytree": [0.7, 1.0]       # column sampling per tree
}

# Stratified CV and grid search
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=45)

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

# Fit on the training set (using the BINARY targets!)
grid.fit(X_train, y_train_bin)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 3))

# Evaluate best model on the test set
best_xgb = grid.best_estimator_
y_pred = best_xgb.predict(X_test)

print("\nTest Accuracy:", round(accuracy_score(y_test_bin, y_pred), 3))
print("\nConfusion Matrix:\n", confusion_matrix(y_test_bin, y_pred))
print("\nClassification Report:\n", classification_report(y_test_bin, y_pred))


Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.7}
Best CV accuracy: 0.833

Test Accuracy: 0.796

Confusion Matrix:
 [[23  7]
 [ 4 20]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.77      0.81        30
           1       0.74      0.83      0.78        24

    accuracy                           0.80        54
   macro avg       0.80      0.80      0.80        54
weighted avg       0.80      0.80      0.80        54



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [12:23:40] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
